In [0]:
%run ../00_utils

In [0]:
from pyspark.sql.window import Window

RAW_COLS = ["trade_date", "symbol", "sector_code", "company_name", "open_raw", "high_raw",
            "low_raw", "close_raw", "volume_raw", "ldcp_raw"]

def num(c):
    return F.expr(f"try_cast(trim({c}) AS DOUBLE)")

def build_silver(start_date, end_date, batch_id):
    b = (spark.table("workspace.bronze.mkt_summary")
              .where(F.col("file_trade_date").between(start_date, end_date))
              .where(~F.trim("symbol").contains("-")))               # drop futures

    s = b.select(
        F.trim("symbol").alias("symbol"),
        F.expr("CAST(try_to_timestamp(trim(trade_date), 'ddMMMyyyy') AS DATE)").alias("trade_date"),
        F.trim("sector_code").alias("sector_code"),
        F.trim("company_name").alias("company_name"),
        num("open_raw").alias("open"),
        num("high_raw").alias("high"),
        num("low_raw").alias("low"),
        num("close_raw").alias("close"),
        num("volume_raw").cast("bigint").alias("volume"),
        num("ldcp_raw").alias("ldcp"),
        "file_trade_date", "source_file",
        F.col("batch_id").alias("bronze_batch_id"),
        F.col("load_timestamp").alias("bronze_load_ts"),
        F.concat_ws("|", *RAW_COLS).alias("raw_record"))

    # Why a row is bad (NULL = row is fine)
    s = s.withColumn("reason",
        F.when((F.col("symbol").isNull()) | (F.col("symbol") == ""), "missing symbol")
         .when(F.col("trade_date").isNull(), "trade_date not parseable")
         .when(F.col("trade_date") != F.col("file_trade_date"), "trade_date differs from file date")
         .when(F.col("close").isNull(), "close not numeric")
         .when(F.col("open").isNull() | F.col("high").isNull() | F.col("low").isNull(), "OHLC not numeric")
         .when(F.col("volume").isNull(), "volume not numeric"))

    bad = s.where(F.col("reason").isNotNull())
    good = s.where(F.col("reason").isNull())

    # One row per (symbol, trade_date): keep the latest Bronze load
    w = Window.partitionBy("symbol", "trade_date").orderBy(F.col("bronze_load_ts").desc())
    good = (good.withColumn("rn", F.row_number().over(w)).where("rn = 1").drop("rn")
                .withColumn("is_traded", F.col("volume") > 0)
                .withColumn("is_ohlc_valid",
                    (F.col("low") > 0) & (F.col("low") <= F.col("high")) &
                    F.col("open").between(F.col("low"), F.col("high")) &
                    F.col("close").between(F.col("low"), F.col("high")))
                .withColumn("row_hash", F.sha2(F.concat_ws("|",
                    "sector_code", "company_name", "open", "high", "low", "close", "volume", "ldcp"), 256))
                .withColumn("batch_id", F.lit(batch_id)))
    return good, bad

In [0]:
def load_silver(start_date, end_date, load_type):
    batch_id = f"silver_{dt.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
    param = f"{start_date}..{end_date}"
    start = dt.datetime.now()
    try:
        good, bad = build_silver(start_date, end_date, batch_id)

        # Quarantine (idempotent: clear this range first)
        spark.sql(f"""DELETE FROM workspace.ops.quarantine_mkt_summary
                      WHERE layer = 'silver' AND file_trade_date BETWEEN '{start_date}' AND '{end_date}'""")
        n_bad = bad.count()
        if n_bad > 0:
            (bad.select("raw_record", "reason", F.lit("silver").alias("layer"), "source_file",
                        "file_trade_date", F.lit(batch_id).alias("batch_id"),
                        F.current_timestamp().alias("load_timestamp"))
                .write.mode("append").saveAsTable("workspace.ops.quarantine_mkt_summary"))

        good.createOrReplaceTempView("silver_src")
        m = spark.sql("""
            MERGE INTO workspace.silver.daily_prices t
            USING silver_src s
            ON t.symbol = s.symbol AND t.trade_date = s.trade_date
            WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET
                t.sector_code = s.sector_code, t.company_name = s.company_name,
                t.open = s.open, t.high = s.high, t.low = s.low, t.close = s.close,
                t.volume = s.volume, t.ldcp = s.ldcp,
                t.is_traded = s.is_traded, t.is_ohlc_valid = s.is_ohlc_valid,
                t.row_hash = s.row_hash, t.source_file = s.source_file,
                t.bronze_batch_id = s.bronze_batch_id, t.batch_id = s.batch_id,
                t.updated_timestamp = current_timestamp()
            WHEN NOT MATCHED THEN INSERT
                (symbol, trade_date, sector_code, company_name, open, high, low, close, volume, ldcp,
                 is_traded, is_ohlc_valid, row_hash, source_file, bronze_batch_id, batch_id,
                 load_timestamp, updated_timestamp)
            VALUES
                (s.symbol, s.trade_date, s.sector_code, s.company_name, s.open, s.high, s.low, s.close,
                 s.volume, s.ldcp, s.is_traded, s.is_ohlc_valid, s.row_hash, s.source_file,
                 s.bronze_batch_id, s.batch_id, current_timestamp(), NULL)
        """).collect()[0]

        inserted, updated = m["num_inserted_rows"], m["num_updated_rows"]
        write_log(batch_id, "bronze_to_silver", load_type, param, start, dt.datetime.now(),
                  "SUCCESS", rows_inserted=inserted, rows_updated=updated, rows_quarantined=n_bad)
        print(f"Done. batch={batch_id}, inserted={inserted}, updated={updated}, quarantined={n_bad}")
    except Exception as e:
        write_log(batch_id, "bronze_to_silver", load_type, param, start, dt.datetime.now(),
                  "FAILURE", error_message=str(e)[:1000])
        print("FAILED:", str(e)[:300])
    return batch_id